# ANN and MLP Architecture
## Turn individual computations into a complete model

Imagine a greenhouse controller receiving two measurements: soil dryness and air warmth. A single score combines dryness and warmth in one fixed way. But perhaps useful intermediate signals include their combined level, which reading is larger, and combinations of those signals. How can several neurons work together while keeping their roles clear?

We need a design that specifies who receives which values and what each stage produces. That design is the network's **architecture**. This lesson builds one small architecture by hand, follows actual numbers through it, and counts exactly what must be learned.

A neuron combines inputs with adjustable weights and a bias, then applies an activation function to produce its response. Nonlinear activations let a network change its response across different input regions instead of remaining a single weighted-sum model. Read the [code companion](04_ANN_and_MLP_Architecture.ipynb) afterward to construct and inspect the same model in PyTorch.

## From one neuron to a layer

A neuron receives inputs, computes a weighted sum plus a bias, and applies an activation. A **layer** groups several neurons that operate on the same incoming representation. Each neuron has its own weights and bias, so the neurons can respond differently even though they receive the same inputs.

Suppose a layer receives dryness $x_1$ and warmth $x_2$. Three neurons could produce three different scores:

$$z_1=x_1+x_2,\qquad z_2=x_1-x_2-0.5,\qquad z_3=-x_1+x_2+0.25.$$

Here $z_1,z_2,z_3$ are the three pre-activation scores. The subscripts identify neurons, not examples. Applying ReLU separately gives hidden outputs $h_j=\max(0,z_j)$ for neuron index $j=1,2,3$. ReLU keeps positive scores and replaces negative ones with zero.

Each neuron has two weights and one bias, so the layer stores nine parameters in total. A weight may happen to be zero; that does not remove it from the parameter count if it remains an adjustable entry.

A **fully connected**, or **dense**, layer gives every output neuron an adjustable connection to every input feature. Dense describes the allowed connections. It does not require every learned weight or every activation to be nonzero. Neurons within this ordinary layer do not feed each other sequentially; they receive the preceding layer's values together.

## What ANN, feedforward, and MLP mean

An **artificial neural network (ANN)** is a broad family of models built from connected parameterized computations. An ANN can have many structures. The particular structure here sends information from input to output without a recurrent loop; this is a **feedforward network**.

A **multilayer perceptron (MLP)** is a feedforward network made from dense layers, normally with nonlinear activations between them. Despite the historical name “perceptron,” a modern MLP generally uses differentiable or piecewise differentiable activations such as ReLU, rather than the hard threshold used by the classical perceptron.

Our example is a $2\to3\to2\to1$ MLP:

```text
Input values       Hidden layer 1       Hidden layer 2       Output
2 features    ──▶   3 units + ReLU  ──▶  2 units + ReLU  ──▶  1 score
                   separate weights    separate weights     separate weights
                   and biases          and biases           and bias
```

The **input layer** means the supplied feature values; it has no learned parameters in this model. A **hidden layer** is an intermediate learned representation, rather than the original input or the final prediction. Hidden does not mean impossible to inspect. The **output layer** maps the final hidden representation to the values the task needs.

We will call this network two hidden layers and three parameterized layers. Some descriptions count depth differently, so stating both avoids ambiguity. **Width** is the number of units in a layer, here 3 and 2 for the hidden layers. Batch size, number of training examples, and number of training epochs are different quantities; none is the width or depth.

## Follow one example all the way through

Use sensor readings on a scale from 0 to 1: dryness $x_1=0.8$ and warmth $x_2=0.4$, where larger values mean drier soil and warmer air. The numbers below are hand-chosen for arithmetic, not learned or validated greenhouse decisions.

The first layer computes:

| Unit | Weighted sum plus bias | Score | After ReLU |
| --- | --- | ---: | ---: |
| 1 | $0.8+0.4+0$ | 1.20 | 1.20 |
| 2 | $0.8-0.4-0.5$ | -0.10 | 0.00 |
| 3 | $-0.8+0.4+0.25$ | -0.15 | 0.00 |

The new representation is $h=(1.2,0,0)$. This is one example with three hidden values, not three examples. The second hidden layer receives these three values rather than the original two readings. Let its scores be

$$u_1=h_1-h_2+0.5h_3-0.2,\qquad u_2=0.5h_1+h_2-h_3+0.1.$$

$h_1,h_2,h_3$ are the preceding hidden outputs, and $u_1,u_2$ are the new scores. Substitution gives $u_1=1.2-0+0-0.2=1.0$ and $u_2=0.6+0-0+0.1=0.7$. Applying ReLU gives $g=(1.0,0.7)$, where $g$ names the second hidden representation.

Finally, define the output score

$$s=2g_1-g_2-0.5=2(1.0)-0.7-0.5=0.8.$$

$s$ is the final scalar score. Each layer has transformed the description available to the next. This is a **forward pass**: computing outputs with the current parameters. No learning takes place merely because we ran it.

If this were a trained binary classifier, sigmoid could turn its final score into a class-1 probability estimate. Here sigmoid of 0.8 is about 0.690, but that number has no validated probability meaning because we invented the parameters. Keeping the output as a raw score lets us inspect the architecture without pretending that it solves a real watering task.

## Connect the layers without losing track of shapes

For a batch of $B$ examples, a dense layer computes

$$Z=HW^\mathsf{T}+b,\qquad A=\phi(Z).$$

$H$ is the matrix of incoming values, one example per row. $W$ is the weight matrix, one output neuron's weights per row. The superscript $\mathsf{T}$ means transpose, which exchanges rows and columns. $b$ is the vector of output-neuron biases, reused for every example. $Z$ holds the pre-activation scores, $\phi$ is the elementwise activation, and $A$ holds the resulting outputs. At the first layer, $H$ is simply the input matrix $X$.

If a layer has $m$ inputs and $n$ outputs, PyTorch stores $W$ with shape $(n,m)$ and $b$ with shape $(n,)$. The input has shape $(B,m)$, so multiplication by $W^\mathsf{T}$ yields $(B,n)$. The next layer must accept those $n$ values.

| Stage in our network | Weight shape | Bias shape | Batch output shape |
| --- | --- | --- | --- |
| Input | — | — | $(B,2)$ |
| Dense 2 → 3, then ReLU | $(3,2)$ | $(3,)$ | $(B,3)$ |
| Dense 3 → 2, then ReLU | $(2,3)$ | $(2,)$ | $(B,2)$ |
| Dense 2 → 1 | $(1,2)$ | $(1,)$ | $(B,1)$ |

The same parameters serve every batch row. Adding more examples increases computation and the memory used for intermediate values, but does not add model parameters. ReLU changes the values without changing the shape.

The output of one layer determines the next layer's input size. A common construction error is to write `Linear(2, 3)` followed by `Linear(2, 1)`: the second layer needs three inputs, not two.

## Count exactly what is adjustable

A dense layer with $m$ input values and $n$ output units has

$$P=mn+n=(m+1)n$$

parameters when a bias is enabled. $P$ is the total count, $mn$ counts all input-to-output weights, and the final $n$ counts one bias per output neuron. Without a bias, only $mn$ remains.

For our model:

| Layer | Weights | Biases | Total |
| --- | ---: | ---: | ---: |
| 2 → 3 | $2\times3=6$ | 3 | 9 |
| 3 → 2 | $3\times2=6$ | 2 | 8 |
| 2 → 1 | $2\times1=2$ | 1 | 3 |
| Whole network | 14 | 6 | **20** |

The two ReLU modules contribute zero learned parameters. “More modules” does not necessarily mean “more parameters.” Some other modules can have learned parameters, so inspect rather than assume.

With 32-bit floating-point storage, 20 parameter values use $20\times4=80$ bytes for the numerical parameter data alone. This is not the model's total memory use: tensor objects, intermediate outputs, gradients, and optimizer state add overhead or additional storage. Parameter count is a useful architectural measurement, not a complete runtime or memory benchmark.

## Width and depth change different things

A wider hidden layer provides more intermediate quantities at the same stage. A deeper model can transform those quantities in additional stages. Neither change guarantees more useful features or better predictions.

For two input features and one output, compare these designs:

| Architecture | Hidden layers | Parameter calculation | Total |
| --- | ---: | --- | ---: |
| 2 → 1 | 0 | $(2+1)1$ | 3 |
| 2 → 4 → 1 | 1 | $(2+1)4+(4+1)1$ | 17 |
| 2 → 8 → 1 | 1 | $(2+1)8+(8+1)1$ | 33 |
| 2 → 4 → 4 → 1 | 2 | $(2+1)4+(4+1)4+(4+1)1$ | 37 |

Increasing width affects both the incoming and outgoing weights. Increasing depth adds transformations and additional parameter groups. **Capacity** describes a model's ability to represent a range of functions; parameter count is one rough indicator, not a complete description of that range or of learnability.

A narrow hidden layer can act as a **bottleneck**, forcing later computation to rely on fewer values. Fewer dimensions can discard useful information, but they may also be sufficient for the task. The number of units alone does not tell us what information survives. Nonlinearities may collapse different inputs to identical hidden values, too.

When choosing an architecture for a real task, distinguish **expressiveness** (a suitable parameter setting exists), **optimization** (training finds useful parameters), and **generalization** (the result works on unseen examples). Our hand-set network demonstrates its computation, not the last two properties. When experimenting with learned models, use validation data for architecture choices and keep the test set for final evaluation.

## Choose the output shape from the task

Hidden width is a design choice, while the output size must agree with what we want to predict. The architecture must specify both how many values to produce and what they mean.

| Task | Common output shape for batch size $B$ | Meaning |
| --- | --- | --- |
| Predict one unrestricted number | $(B,1)$ | One raw regression output per example |
| Binary classification using one logit | $(B,1)$ | One class-1 score per example |
| Exactly one of $K$ classes | $(B,K)$ | One competing score per class |
| $K$ independently possible labels | $(B,K)$ | One independent score per label |
| Predict two continuous measurements | $(B,2)$ | Two regression outputs per example |

$K$ is the number of classes or labels. Matching shape is necessary but not sufficient: two tasks can use the same shape with different output interpretations and losses. A logit is a raw class score before probability conversion. For classification losses that accept logits, retain these scores in the model. For probability display, sigmoid converts one binary score to a class-1 estimate, while softmax normalizes competing class scores to sum to one.

The final mapping is often called an **output head**. Different heads can use a shared hidden representation. The companion illustrates one-value and three-value heads without claiming that an untrained hidden representation is useful for either task.

## How this appears in PyTorch

An `nn.Sequential` model passes each module's output directly to the next module. It is a readable way to express a simple chain of dense layers and activations. An `nn.Module` subclass with a `forward` method makes named intermediate values and more complex paths explicit. Both forms can express the same architecture and compute the same function when their parameters match.

The architecture is the arrangement and dimensions. The parameters are the numerical weights and biases stored within it. Two models with identical architecture can produce different outputs because their parameters differ. A random initialization provides starting values for training; it is not a trained model.

`model.eval()` switches certain modules, such as dropout and batch normalization, to their evaluation behavior. Our dense/ReLU model has no such behavior change. `torch.no_grad()` instead stops recording operations for gradient computation. Neither command trains a model, and they are not interchangeable.

In this lesson, named layers make the path easy to inspect. Printing weights alone does not reveal a hidden unit's practical meaning; inspect its responses and the task before assigning it a label.

## Uses and limitations

MLPs can model relationships among fixed-size feature vectors and can serve as components within larger systems. They do not automatically exploit spatial neighborhoods in images or sequence order in text. Flattening a large image into one vector can make the first dense layer very large: a 32 × 32 RGB image has 3,072 values, and connecting them to 128 units requires $(3072+1)128=393,344$ parameters in that layer alone. RGB means three color channels: red, green, and blue.

This motivates architectures with more specific structure, including convolutional neural networks (CNNs), which use local connections for images, and sequence models that account for order. Transformers also contain feedforward components, but an MLP by itself is not a transformer or a complete generative model.

Common mistakes include confusing neurons with examples, counting activations as parameterized layers without checking, omitting nonlinearities between dense layers, and changing a hidden width without adjusting the next layer's input size. A larger architecture may overfit, cost more to run, or be harder to train; it does not guarantee a better result.

## Recap and self-check

An MLP chains dense layers and nonlinear activations. Each layer re-expresses the preceding layer's values, while the output head matches the prediction task. Shape tracing verifies that the pieces connect; parameter counting measures how many numerical values the model stores and can learn.

- How many parameters are in a biased dense layer with 3 inputs and 5 outputs?
- For a batch of 10 examples, what shape follows a 3 → 5 layer and ReLU?
- Does doubling batch size double the model's parameter count?
- In a 2 → 4 → 1 MLP, what input size must the final dense layer accept?

Check: $(3+1)5=20$ parameters; shape $(10,5)$; no, parameters are shared; and the final layer takes four input values.